In [1]:
# Cell 1: Setup & Constants
# Notebook 07: Dim_Organisation — Gold_SalesOps_Dim_Organisation
# Source: Organisation (Silver)
# Grain: OrganisationId (one row per WTW team/office/entity)

from pyspark.sql import functions as F

SILVER_BASE = "abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo"

print("Setup complete.")
print(f"Silver base: {SILVER_BASE}")

StatementMeta(, ca6a0cdf-b984-433c-89b1-9568e31bc37f, 3, Finished, Available, Finished, False)

Setup complete.
Silver base: abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo


In [2]:
# Cell 2: Load Organisation table

df_org = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Organisation")
    .filter(F.col("IsDeleted") == False)
)

total = df_org.count()
distinct = df_org.select("OrganisationId").distinct().count()

print(f"Organisation rows:        {total:,}")
print(f"Distinct OrganisationId:  {distinct:,}")
print(f"OrganisationId unique?    {'YES' if distinct == total else 'NO — DUPLICATES EXIST'}")

StatementMeta(, ca6a0cdf-b984-433c-89b1-9568e31bc37f, 4, Finished, Available, Finished, False)

Organisation rows:        58,494
Distinct OrganisationId:  58,494
OrganisationId unique?    YES


In [3]:
# Cell 3: Select Final Columns

dim_organisation = df_org.select(
    # Key
    "OrganisationId",
    "SourceId",
    "OrganisationKey",
    # Name
    "Organisation",
    # Hierarchy
    "Level1",
    "Level2",
    "Level3",
    "Level4",
    "Level1Code",
    "Level2Code",
    "Level3Code",
    "Level4Code",
    # Parent
    "ParentId",
)

row_count = dim_organisation.count()
print(f"Dim_Organisation rows: {row_count:,}")

# NULL counts for key columns
print("\nKey column NULL counts:")
print("-" * 55)
for col_name in ["Organisation", "Level1", "Level2", "Level3", "Level4", "ParentId"]:
    null_count = dim_organisation.filter(F.col(col_name).isNull()).count()
    pct = null_count / row_count * 100 if row_count > 0 else 0
    print(f"  {col_name:<35} {null_count:>12,}  ({pct:5.1f}%)")

StatementMeta(, ca6a0cdf-b984-433c-89b1-9568e31bc37f, 5, Finished, Available, Finished, False)

Dim_Organisation rows: 58,494

Key column NULL counts:
-------------------------------------------------------
  Organisation                                  58  (  0.1%)
  Level1                                    52,599  ( 89.9%)
  Level2                                    52,599  ( 89.9%)
  Level3                                    52,606  ( 89.9%)
  Level4                                    56,469  ( 96.5%)
  ParentId                                  24,089  ( 41.2%)


In [4]:
# Cell 4: Write to Gold Lakehouse

dim_organisation.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("Gold_SalesOps_Dim_Organisation")

final_count = spark.read.table("Gold_SalesOps_Dim_Organisation").count()
print(f"Gold_SalesOps_Dim_Organisation written: {final_count:,} rows")

StatementMeta(, ca6a0cdf-b984-433c-89b1-9568e31bc37f, 6, Finished, Available, Finished, False)

Gold_SalesOps_Dim_Organisation written: 58,494 rows
